# Equipment classification

Classify failure/normal labels using measurements and machine type. [Data notes](../datasets/README.md).


In [ ]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if not (root / "patient_data.csv").exists():
    root = root.parent
data = pd.read_csv(root / "datasets/equipment.csv")
print(data.shape)
print(data["label"].value_counts())
data.head()


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = data.drop(columns="label")
y = data["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
numeric_columns = X.select_dtypes(include="number").columns.tolist()
category_columns = X.select_dtypes(exclude="number").columns.tolist()
preprocess = ColumnTransformer([
    ("numbers", StandardScaler(), numeric_columns),
    ("categories", OneHotEncoder(handle_unknown="ignore"), category_columns),
])
model = make_pipeline(preprocess, RandomForestClassifier(n_estimators=80, max_depth=6, random_state=42, n_jobs=1))
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, prediction))


In [ ]:
import matplotlib.pyplot as plt

data.select_dtypes(include="number").hist(figsize=(9, 6), bins=20)
plt.tight_layout()
plt.show()


In [ ]:
feature_names = model[0].get_feature_names_out()
importance = pd.Series(model[-1].feature_importances_, index=feature_names)
importance.sort_values().plot.barh(title="Equipment forest feature importance")
plt.tight_layout()
plt.show()
